# **Landhills Winery Case (Optimization Problem)**
## Questions 1 (Base model)
## Question 2 (Merlot quantity discount extension of base model)

## **Intalling and importing packages**

In [2]:
!pip install gurobipy
from gurobipy import *

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 63.7 MB/s eta 0:00:00


# **Question 1: Base model (no siscount; for question 1)**

First, we solve the base optimization problem without any quantity discounts.

In [3]:
# Create base model
m_base = Model("Landhills_Base")

# Grape types and wine products
grapes = ["CS_SB_2011", "CS_SLO_2010", "CS_SLO_2011", "Merlot_SB_2010"]
wines = ["Vintage_CS_2011_SB", "Vintage_CS_2010_SLO", "Vintage_CS_2011_SLO", "NonVintage_CS", "NonVintage_Merlot"]

# Parameters
acidity = {"CS_SB_2011": 0.35, "CS_SLO_2010": 0.75, "CS_SLO_2011": 0.55, "Merlot_SB_2010": 0.25}
sugar = {"CS_SB_2011": 0.12, "CS_SLO_2010": 0.25, "CS_SLO_2011": 0.30, "Merlot_SB_2010": 0.08}
alcohol = {"CS_SB_2011": 13.5, "CS_SLO_2010": 15.3, "CS_SLO_2011": 11.5, "Merlot_SB_2010": 15.7}
availability = {"CS_SB_2011": 50000, "CS_SLO_2010": 60000, "CS_SLO_2011": 30000, "Merlot_SB_2010": 200000}
cost_per_bottle = {"CS_SB_2011": 2.35, "CS_SLO_2010": 2.60, "CS_SLO_2011": 2.10, "Merlot_SB_2010": 1.55}
price = {"Vintage_CS_2011_SB": 9.00, "Vintage_CS_2010_SLO": 9.00, "Vintage_CS_2011_SLO": 9.00, "NonVintage_CS": 5.50, "NonVintage_Merlot": 2.95}

# Grape classifications
is_cabernet = {"CS_SB_2011": 1, "CS_SLO_2010": 1, "CS_SLO_2011": 1, "Merlot_SB_2010": 0}
is_merlot = {"CS_SB_2011": 0, "CS_SLO_2010": 0, "CS_SLO_2011": 0, "Merlot_SB_2010": 1}
is_2011 = {"CS_SB_2011": 1, "CS_SLO_2010": 0, "CS_SLO_2011": 1, "Merlot_SB_2010": 0}
is_2010 = {"CS_SB_2011": 0, "CS_SLO_2010": 1, "CS_SLO_2011": 0, "Merlot_SB_2010": 1}
is_santa_barbara = {"CS_SB_2011": 1, "CS_SLO_2010": 0, "CS_SLO_2011": 0, "Merlot_SB_2010": 1}
is_san_luis_obispo = {"CS_SB_2011": 0, "CS_SLO_2010": 1, "CS_SLO_2011": 1, "Merlot_SB_2010": 0}

# Decision variables
x_base = m_base.addVars(grapes, wines, vtype=GRB.CONTINUOUS, lb=0, name="x")

# Constraints
# 1. Availability
m_base.addConstrs((quicksum(x_base[i,j] for j in wines) <= availability[i] for i in grapes), name="Availability")

# 2. Acidity
m_base.addConstr(quicksum(acidity[i] * x_base[i, "Vintage_CS_2011_SB"] for i in grapes) <= 0.7 * quicksum(x_base[i, "Vintage_CS_2011_SB"] for i in grapes))
m_base.addConstr(quicksum(acidity[i] * x_base[i, "Vintage_CS_2010_SLO"] for i in grapes) <= 0.7 * quicksum(x_base[i, "Vintage_CS_2010_SLO"] for i in grapes))
m_base.addConstr(quicksum(acidity[i] * x_base[i, "Vintage_CS_2011_SLO"] for i in grapes) <= 0.7 * quicksum(x_base[i, "Vintage_CS_2011_SLO"] for i in grapes))
m_base.addConstr(quicksum(acidity[i] * x_base[i, "NonVintage_CS"] for i in grapes) <= 0.7 * quicksum(x_base[i, "NonVintage_CS"] for i in grapes))
m_base.addConstr(quicksum(acidity[i] * x_base[i, "NonVintage_Merlot"] for i in grapes) <= 0.3 * quicksum(x_base[i, "NonVintage_Merlot"] for i in grapes))

# 3. Sugar
m_base.addConstr(quicksum(sugar[i] * x_base[i, "Vintage_CS_2011_SB"] for i in grapes) <= 0.2 * quicksum(x_base[i, "Vintage_CS_2011_SB"] for i in grapes))
m_base.addConstr(quicksum(sugar[i] * x_base[i, "Vintage_CS_2010_SLO"] for i in grapes) <= 0.2 * quicksum(x_base[i, "Vintage_CS_2010_SLO"] for i in grapes))
m_base.addConstr(quicksum(sugar[i] * x_base[i, "Vintage_CS_2011_SLO"] for i in grapes) <= 0.2 * quicksum(x_base[i, "Vintage_CS_2011_SLO"] for i in grapes))
m_base.addConstr(quicksum(sugar[i] * x_base[i, "NonVintage_CS"] for i in grapes) <= 0.3 * quicksum(x_base[i, "NonVintage_CS"] for i in grapes))

# 4. Alcohol (10-15%)
m_base.addConstrs((quicksum(alcohol[i] * x_base[i,j] for i in grapes) >= 10.0 * quicksum(x_base[i,j] for i in grapes) for j in wines), name="Alcohol_Lower")
m_base.addConstrs((quicksum(alcohol[i] * x_base[i,j] for i in grapes) <= 15.0 * quicksum(x_base[i,j] for i in grapes) for j in wines), name="Alcohol_Upper")

# 5. Varietal requirements (>= 75%)
m_base.addConstr(quicksum(is_cabernet[i] * x_base[i, "Vintage_CS_2011_SB"] for i in grapes) >= 0.75 * quicksum(x_base[i, "Vintage_CS_2011_SB"] for i in grapes))
m_base.addConstr(quicksum(is_cabernet[i] * x_base[i, "Vintage_CS_2010_SLO"] for i in grapes) >= 0.75 * quicksum(x_base[i, "Vintage_CS_2010_SLO"] for i in grapes))
m_base.addConstr(quicksum(is_cabernet[i] * x_base[i, "Vintage_CS_2011_SLO"] for i in grapes) >= 0.75 * quicksum(x_base[i, "Vintage_CS_2011_SLO"] for i in grapes))
m_base.addConstr(quicksum(is_cabernet[i] * x_base[i, "NonVintage_CS"] for i in grapes) >= 0.75 * quicksum(x_base[i, "NonVintage_CS"] for i in grapes))
m_base.addConstr(quicksum(is_merlot[i] * x_base[i, "NonVintage_Merlot"] for i in grapes) >= 0.75 * quicksum(x_base[i, "NonVintage_Merlot"] for i in grapes))

# 6. Vintage year requirements (>= 95%)
m_base.addConstr(quicksum(is_2011[i] * x_base[i, "Vintage_CS_2011_SB"] for i in grapes) >= 0.95 * quicksum(x_base[i, "Vintage_CS_2011_SB"] for i in grapes))
m_base.addConstr(quicksum(is_2010[i] * x_base[i, "Vintage_CS_2010_SLO"] for i in grapes) >= 0.95 * quicksum(x_base[i, "Vintage_CS_2010_SLO"] for i in grapes))
m_base.addConstr(quicksum(is_2011[i] * x_base[i, "Vintage_CS_2011_SLO"] for i in grapes) >= 0.95 * quicksum(x_base[i, "Vintage_CS_2011_SLO"] for i in grapes))

# 7. Viticulture area requirements (>= 85%)
m_base.addConstr(quicksum(is_santa_barbara[i] * x_base[i, "Vintage_CS_2011_SB"] for i in grapes) >= 0.85 * quicksum(x_base[i, "Vintage_CS_2011_SB"] for i in grapes))
m_base.addConstr(quicksum(is_san_luis_obispo[i] * x_base[i, "Vintage_CS_2010_SLO"] for i in grapes) >= 0.85 * quicksum(x_base[i, "Vintage_CS_2010_SLO"] for i in grapes))
m_base.addConstr(quicksum(is_san_luis_obispo[i] * x_base[i, "Vintage_CS_2011_SLO"] for i in grapes) >= 0.85 * quicksum(x_base[i, "Vintage_CS_2011_SLO"] for i in grapes))

# Objective function: profit maximization
obj_base = quicksum(price[j] * quicksum(x_base[i,j] for i in grapes) for j in wines) - quicksum(cost_per_bottle[i] * quicksum(x_base[i,j] for j in wines) for i in grapes)
m_base.setObjective(obj_base, GRB.MAXIMIZE)

# Solving
m_base.optimize()

# Results
print("\n" + "="*80)
print("QUESTION 1: BASE MODEL RESULTS")
print("="*80)
print(f"\nOptimal Profit: ${m_base.objVal:,.2f}\n")

print("Production by wine:")
for j in wines:
    bottles = sum(x_base[i,j].X for i in grapes)
    if bottles > 0.1:
        print(f"{j}: {bottles:,.0f} bottles")

print(f"\nMerlot procured: {sum(x_base['Merlot_SB_2010',j].X for j in wines):,.0f} bottles")
print(f"Merlot cost: ${sum(x_base['Merlot_SB_2010',j].X for j in wines) * 1.55:,.2f}")

Restricted license - for non-production use only - expires 2027-11-29
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 34 rows, 20 columns and 139 nonzeros (Max)
Model fingerprint: 0xc34ff76b
Model has 20 linear objective coefficients
Coefficient statistics:
  Matrix range     [5e-02, 6e+00]
  Objective range  [4e-01, 7e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [3e+04, 2e+05]

Presolve removed 9 rows and 2 columns
Presolve time: 0.02s
Presolved: 25 rows, 18 columns, 119 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    7.9000000e+30   1.050000e+30   7.900000e+00      0s
      13    8.0991139e+05   0.000000e+00   0.000000e+00      0s

Solved in 13 iterations and 0.03 seconds (0.00 work units)
Optimal objective  8.099113856e+05

QUE

# **Question 2: Extension with Merlot quantity discount**

Now we modify the base model to incorporate an all-units quantity discount for Merlot:
- Standard price: 1.55/bottle for orders < 150,000 bottles
- Discount price: 1.10/bottle for orders >= 150,000 bottles

This requires binary variables following the mathematical formulation provided.

In [4]:
# Creating model with discount
m_discount = Model("Landhills_Discount")

# Same parameters as base
# (so grapes, wines, acidity, sugar etc already defined)

# Merlot discount parameters
merlot_price_high = 1.55
merlot_price_low = 1.10
merlot_threshold = 150000

# Decision variables
# Original variables: x[i,j] = bottles of grape i in wine j
x = m_discount.addVars(grapes, wines, vtype=GRB.CONTINUOUS, lb=0, name="x")

# Variables for quantity discount
W = m_discount.addVar(vtype=GRB.CONTINUOUS, lb=0, name="W")  # Total Merlot procured
R1 = m_discount.addVar(vtype=GRB.CONTINUOUS, lb=0, name="R1")  # Amount at high price
R2 = m_discount.addVar(vtype=GRB.CONTINUOUS, lb=0, name="R2")  # Amount at low price
Y1 = m_discount.addVar(vtype=GRB.BINARY, name="Y1")  # Binary: 1 if high price tier
Y2 = m_discount.addVar(vtype=GRB.BINARY, name="Y2")  # Binary: 1 if low price tier

# All original constraints (same as base model)
# 1. Availability for non-Merlot grapes
for i in ["CS_SB_2011", "CS_SLO_2010", "CS_SLO_2011"]:
    m_discount.addConstr(quicksum(x[i,j] for j in wines) <= availability[i])

# All quality and regulatory constraints (same as base)
m_discount.addConstr(quicksum(acidity[i] * x[i, "Vintage_CS_2011_SB"] for i in grapes) <= 0.7 * quicksum(x[i, "Vintage_CS_2011_SB"] for i in grapes))
m_discount.addConstr(quicksum(acidity[i] * x[i, "Vintage_CS_2010_SLO"] for i in grapes) <= 0.7 * quicksum(x[i, "Vintage_CS_2010_SLO"] for i in grapes))
m_discount.addConstr(quicksum(acidity[i] * x[i, "Vintage_CS_2011_SLO"] for i in grapes) <= 0.7 * quicksum(x[i, "Vintage_CS_2011_SLO"] for i in grapes))
m_discount.addConstr(quicksum(acidity[i] * x[i, "NonVintage_CS"] for i in grapes) <= 0.7 * quicksum(x[i, "NonVintage_CS"] for i in grapes))
m_discount.addConstr(quicksum(acidity[i] * x[i, "NonVintage_Merlot"] for i in grapes) <= 0.3 * quicksum(x[i, "NonVintage_Merlot"] for i in grapes))
m_discount.addConstr(quicksum(sugar[i] * x[i, "Vintage_CS_2011_SB"] for i in grapes) <= 0.2 * quicksum(x[i, "Vintage_CS_2011_SB"] for i in grapes))
m_discount.addConstr(quicksum(sugar[i] * x[i, "Vintage_CS_2010_SLO"] for i in grapes) <= 0.2 * quicksum(x[i, "Vintage_CS_2010_SLO"] for i in grapes))
m_discount.addConstr(quicksum(sugar[i] * x[i, "Vintage_CS_2011_SLO"] for i in grapes) <= 0.2 * quicksum(x[i, "Vintage_CS_2011_SLO"] for i in grapes))
m_discount.addConstr(quicksum(sugar[i] * x[i, "NonVintage_CS"] for i in grapes) <= 0.3 * quicksum(x[i, "NonVintage_CS"] for i in grapes))
m_discount.addConstrs((quicksum(alcohol[i] * x[i,j] for i in grapes) >= 10.0 * quicksum(x[i,j] for i in grapes) for j in wines))
m_discount.addConstrs((quicksum(alcohol[i] * x[i,j] for i in grapes) <= 15.0 * quicksum(x[i,j] for i in grapes) for j in wines))
m_discount.addConstr(quicksum(is_cabernet[i] * x[i, "Vintage_CS_2011_SB"] for i in grapes) >= 0.75 * quicksum(x[i, "Vintage_CS_2011_SB"] for i in grapes))
m_discount.addConstr(quicksum(is_cabernet[i] * x[i, "Vintage_CS_2010_SLO"] for i in grapes) >= 0.75 * quicksum(x[i, "Vintage_CS_2010_SLO"] for i in grapes))
m_discount.addConstr(quicksum(is_cabernet[i] * x[i, "Vintage_CS_2011_SLO"] for i in grapes) >= 0.75 * quicksum(x[i, "Vintage_CS_2011_SLO"] for i in grapes))
m_discount.addConstr(quicksum(is_cabernet[i] * x[i, "NonVintage_CS"] for i in grapes) >= 0.75 * quicksum(x[i, "NonVintage_CS"] for i in grapes))
m_discount.addConstr(quicksum(is_merlot[i] * x[i, "NonVintage_Merlot"] for i in grapes) >= 0.75 * quicksum(x[i, "NonVintage_Merlot"] for i in grapes))
m_discount.addConstr(quicksum(is_2011[i] * x[i, "Vintage_CS_2011_SB"] for i in grapes) >= 0.95 * quicksum(x[i, "Vintage_CS_2011_SB"] for i in grapes))
m_discount.addConstr(quicksum(is_2010[i] * x[i, "Vintage_CS_2010_SLO"] for i in grapes) >= 0.95 * quicksum(x[i, "Vintage_CS_2010_SLO"] for i in grapes))
m_discount.addConstr(quicksum(is_2011[i] * x[i, "Vintage_CS_2011_SLO"] for i in grapes) >= 0.95 * quicksum(x[i, "Vintage_CS_2011_SLO"] for i in grapes))
m_discount.addConstr(quicksum(is_santa_barbara[i] * x[i, "Vintage_CS_2011_SB"] for i in grapes) >= 0.85 * quicksum(x[i, "Vintage_CS_2011_SB"] for i in grapes))
m_discount.addConstr(quicksum(is_san_luis_obispo[i] * x[i, "Vintage_CS_2010_SLO"] for i in grapes) >= 0.85 * quicksum(x[i, "Vintage_CS_2010_SLO"] for i in grapes))
m_discount.addConstr(quicksum(is_san_luis_obispo[i] * x[i, "Vintage_CS_2011_SLO"] for i in grapes) >= 0.85 * quicksum(x[i, "Vintage_CS_2011_SLO"] for i in grapes))

# Added constraints for quantity discount
# Constraint 1: R1 <= S1 * Y1
m_discount.addConstr(R1 <= merlot_threshold * Y1, name="Discount_R1")

# Constraint 2: S1 * Y2 <= R2 <= S2 * Y2
m_discount.addConstr(R2 >= merlot_threshold * Y2, name="Discount_R2_lower")
m_discount.addConstr(R2 <= availability["Merlot_SB_2010"] * Y2, name="Discount_R2_upper")

# Constraint 3: Y1 + Y2 = 1
m_discount.addConstr(Y1 + Y2 == 1, name="Discount_Binary")

# Constraint 4: R1 + R2 = W
m_discount.addConstr(R1 + R2 == W, name="Discount_Total")

# Constraint 5: W = sum of Merlot usage across all wines
m_discount.addConstr(W == quicksum(x["Merlot_SB_2010", j] for j in wines), name="Discount_Link")

# Modified objective function
# Revenue from all wines
revenue = quicksum(price[j] * quicksum(x[i,j] for i in grapes) for j in wines)

# Cost of non-Merlot grapes
cost_other_grapes = quicksum(cost_per_bottle[i] * quicksum(x[i,j] for j in wines) for i in ["CS_SB_2011", "CS_SLO_2010", "CS_SLO_2011"])

# Cost of Merlot (we are using here R1 and R2 with different prices)
cost_merlot = merlot_price_high * R1 + merlot_price_low * R2

# Total profit
obj_discount = revenue - cost_other_grapes - cost_merlot
m_discount.setObjective(obj_discount, GRB.MAXIMIZE)

# Solving
m_discount.optimize()

# Results
print("\n" + "="*80)
print("QUESTION 2: MODEL WITH MERLOT QUANTITY DISCOUNT")
print("="*80)
print(f"\nOptimal Profit: ${m_discount.objVal:,.2f}")
print(f"Profit improvement: ${m_discount.objVal - m_base.objVal:+,.2f} ({(m_discount.objVal - m_base.objVal)/m_base.objVal*100:+.1f}%)\n")

print("Production by wine:")
print(f"{'Wine':<30} {'New plan':>15} {'Base plan':>15} {'Change':>15}")
print("-"*75)
for j in wines:
    new_bottles = sum(x[i,j].X for i in grapes)
    old_bottles = sum(x_base[i,j].X for i in grapes)
    if new_bottles > 0.1 or old_bottles > 0.1:
        change = new_bottles - old_bottles
        print(f"{j:<30} {new_bottles:>15,.0f} {old_bottles:>15,.0f} {change:>+15,.0f}")

print("\nMerlot procurement decision:")
print(f"Total Merlot procured (W): {W.X:,.0f} bottles")
print(f"Amount at high price (R1): {R1.X:,.0f} bottles @ ${merlot_price_high}/bottle")
print(f"Amount at low price (R2):  {R2.X:,.0f} bottles @ ${merlot_price_low}/bottle")
print(f"Pricing tier active: {'High ($1.55)' if Y1.X > 0.5 else 'Low ($1.10) - Discount activated'}")
print(f"\nMerlot cost comparison:")
print(f"  Base model:      {sum(x_base['Merlot_SB_2010',j].X for j in wines):>10,.0f} bottles @ $1.55 = ${sum(x_base['Merlot_SB_2010',j].X for j in wines) * 1.55:>10,.2f}")
print(f"  With discount:   {W.X:>10,.0f} bottles @ ${merlot_price_low if Y2.X > 0.5 else merlot_price_high} = ${merlot_price_high * R1.X + merlot_price_low * R2.X:>10,.2f}")
print(f"  Savings:         ${(sum(x_base['Merlot_SB_2010',j].X for j in wines) * 1.55) - (merlot_price_high * R1.X + merlot_price_low * R2.X):>10,.2f}")

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 39 rows, 25 columns and 151 nonzeros (Max)
Model fingerprint: 0x4caf4e39
Model has 22 linear objective coefficients
Variable types: 23 continuous, 2 integer (2 binary)
Coefficient statistics:
  Matrix range     [5e-02, 2e+05]
  Objective range  [4e-01, 9e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 6e+04]

Found heuristic solution: objective 473013.93189
Presolve removed 17 rows and 2 columns
Presolve time: 0.00s
Presolved: 22 rows, 23 columns, 88 nonzeros
Variable types: 22 continuous, 1 integer (1 binary)

Root relaxation: objective 8.670242e+05, 12 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent  